In [1]:
!apt-get install -y bcftools tabix

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libhts3 libhtscodecs2
Suggested packages:
  python3-numpy python3-matplotlib texlive-latex-recommended
The following NEW packages will be installed:
  bcftools libhts3 libhtscodecs2 tabix
0 upgraded, 4 newly installed, 0 to remove and 2 not upgraded.
Need to get 1,491 kB of archives.
After this operation, 4,626 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libhtscodecs2 amd64 1.1.1-3 [53.2 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libhts3 amd64 1.13+ds-2build1 [390 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 bcftools amd64 1.13-1 [697 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tabix amd64 1.13+ds-2build1 [351 kB]
Fetched 1,491 kB in 1s (1,071 kB/s)
Selecting previously unselected package libhtscodecs2:amd64.
(Reading da

In [2]:
!git clone https://github.com/gerardpuchebonjorn/ioannidis
!git clone https://github.com/AI-sandbox/pclai

Cloning into 'ioannidis'...
remote: Enumerating objects: 159, done.
remote: Counting objects: 100% (159/159), done.
remote: Compressing objects: 100% (118/118), done.
remote: Total 159 (delta 95), reused 87 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (159/159), 88.35 KiB | 4.91 MiB/s, done.
Resolving deltas: 100% (95/95), done.
Cloning into 'pclai'...
remote: Enumerating objects: 148, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 148 (delta 19), reused 28 (delta 9), pack-reused 110 (from 2)
Receiving objects: 100% (148/148), 361.00 MiB | 20.35 MiB/s, done.
Resolving deltas: 100% (58/58), done.
Updating files: 100% (82/82), done.


In [3]:
!pip install pandas
%cd ioannidis

/content/ioannidis


In [14]:
%cd /content/ioannidis
!git pull origin dev
!git checkout -f dev
!git pull origin dev
!git log --oneline -3

/content/ioannidis
From https://github.com/gerardpuchebonjorn/ioannidis
 * branch            dev        -> FETCH_HEAD
Updating facd36d..ffd9616
error: Your local changes to the following files would be overwritten by merge:
	lai_pipeline/__pycache__/pipeline.cpython-312.pyc
Please commit your changes or stash them before you merge.
Aborting
Already on 'dev'
Your branch is behind 'origin/dev' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/gerardpuchebonjorn/ioannidis
 * branch            dev        -> FETCH_HEAD
Updating facd36d..ffd9616
Fast-forward
 lai_pipeline/__pycache__/pipeline.cpython-312.pyc | Bin 12157 -> 12208 bytes
 lai_pipeline/pipeline.py                          |   7 ++++---
 2 files changed, 4 insertions(+), 3 deletions(-)
ffd9616 (HEAD -> dev, origin/dev) fix: only prepare reference VCF when impute engine requires it
facd36d feat: adapt pipeline to bundle-based architecture with TSV SNP manifests
9bcb4f9 d

In [15]:
!bcftools index -t /content/chr21.admx.vcf.gz
!bcftools view -h /content/chr21.admx.vcf.gz | head -5

[E::main_vcfindex] the index file exists. Please use '-f' to overwrite /content/chr21.admx.vcf.gz.tbi
##fileformat=VCFv4.2
##FILTER=<ID=PASS,Description="All filters passed">
##source=shapeit5 phase_rare v1.1
##contig=<ID=21>
##INFO=<ID=AC,Number=A,Type=Integer,Description="ALT allele count">


In [17]:
!python cli.py \
  --input-vcf /content/chr21.admx.vcf.gz \
  --workdir /content/workdir \
  --bundle-dir /content/pclai/bundles/pclai_1kg_bundle_cpu \
  --impute-engine none \
  --log-level DEBUG

2026-04-18 09:48:55,906 | INFO     | Logging initialized at level=DEBUG
2026-04-18 09:48:55,906 | INFO     | === LAI HARMONIZATION PIPELINE ===
2026-04-18 09:48:55,906 | INFO     |   input-vcf:              /content/chr21.admx.vcf.gz
2026-04-18 09:48:55,906 | INFO     |   workdir:                /content/workdir
2026-04-18 09:48:55,906 | INFO     |   bundle-dir:             /content/pclai/bundles/pclai_1kg_bundle_cpu
2026-04-18 09:48:55,907 | INFO     |   reference-vcf-template: (none)
2026-04-18 09:48:55,907 | INFO     |   reference-fasta:        (none)
2026-04-18 09:48:55,907 | INFO     |   impute-engine:          none
2026-04-18 09:48:55,907 | INFO     |   beagle-jar:             (none)
2026-04-18 09:48:55,907 | INFO     |   threads:                8
2026-04-18 09:48:55,907 | INFO     |   qc-strict:              False
2026-04-18 09:48:55,907 | INFO     |   log-level:              DEBUG
2026-04-18 09:48:55,907 | INFO     | ==================================
2026-04-18 09:48:55,908 | 

In [18]:
!bcftools view -s HG00551,HG00553,HG00554,HG00637,HG00638,HG00640,HG00641,HG00734,HG00736,HG00737 \
  /content/chr21.admx.vcf.gz \
  -Oz -o /content/chr21.small.vcf.gz
!bcftools index -t /content/chr21.small.vcf.gz

In [19]:
!python cli.py \
  --input-vcf /content/chr21.small.vcf.gz \
  --workdir /content/workdir3 \
  --bundle-dir /content/pclai/bundles/pclai_1kg_bundle_cpu \
  --impute-engine none \
  --allow-other-mismatch \
  --min-exact-match-pct 90.0 \
  --log-level INFO

2026-04-18 10:06:15,883 | INFO     | Logging initialized at level=INFO
2026-04-18 10:06:15,884 | INFO     | === LAI HARMONIZATION PIPELINE ===
2026-04-18 10:06:15,884 | INFO     |   input-vcf:              /content/chr21.small.vcf.gz
2026-04-18 10:06:15,884 | INFO     |   workdir:                /content/workdir3
2026-04-18 10:06:15,884 | INFO     |   bundle-dir:             /content/pclai/bundles/pclai_1kg_bundle_cpu
2026-04-18 10:06:15,884 | INFO     |   reference-vcf-template: (none)
2026-04-18 10:06:15,884 | INFO     |   reference-fasta:        (none)
2026-04-18 10:06:15,884 | INFO     |   impute-engine:          none
2026-04-18 10:06:15,884 | INFO     |   beagle-jar:             (none)
2026-04-18 10:06:15,884 | INFO     |   threads:                8
2026-04-18 10:06:15,884 | INFO     |   qc-strict:              False
2026-04-18 10:06:15,885 | INFO     |   log-level:              INFO
2026-04-18 10:06:15,885 | INFO     | ==================================
2026-04-18 10:06:15,887 | 